In [ ]:
# ============================================================
# Walker2D LSTM PPO (RecurrentPPO) — Jupyter Notebook Script
# ============================================================

# ----------------------------
# 1. Imports
# ----------------------------
import os
import gymnasium as gym
import numpy as np
import torch

from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback
from sb3_contrib import RecurrentPPO

# ----------------------------
# 2. Config
# ----------------------------
ENV_ID = "Walker2d-v4"
SEED = 42
N_ENVS = 8
TOTAL_TIMESTEPS = 20_000_000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# 3. Vectorized Environment
# ----------------------------
env = make_vec_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED
)

# ----------------------------
# 4. LSTM Policy Configuration
# ----------------------------
policy_kwargs = dict(
    lstm_hidden_size=256,
    n_lstm_layers=1,
    shared_lstm=True,
    enable_critic_lstm=True,
    net_arch=dict(pi=[128], vf=[128]),
)

# ----------------------------
# 5. Model Definition
# ----------------------------
model = RecurrentPPO(
    policy="MlpLstmPolicy",
    env=env,
    learning_rate=3e-4,
    n_steps=2048,              # Must be divisible by n_envs
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.003,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    verbose=1,
    seed=SEED,
    device=DEVICE,
)

# ----------------------------
# 6. Checkpointing
# ----------------------------
checkpoint_callback = CheckpointCallback(
    save_freq=500_000,
    save_path="./checkpoints",
    name_prefix="walker2d_lstm"
)

# ----------------------------
# 7. Training
# ----------------------------
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback
)

# ----------------------------
# 8. Save Final Model
# ----------------------------
model.save("walker2d_lstm_final")
print("Model saved as walker2d_lstm_final")

# ----------------------------
# 9. Evaluation (LSTM-safe)
# ----------------------------
eval_env = gym.make(ENV_ID, render_mode="human")

obs, _ = eval_env.reset(seed=SEED)
lstm_states = None
episode_starts = np.ones((1,), dtype=bool)

episode_reward = 0.0

for step in range(2000):
    action, lstm_states = model.predict(
        obs,
        state=lstm_states,
        episode_start=episode_starts,
        deterministic=True
    )

    obs, reward, terminated, truncated, _ = eval_env.step(action)
    episode_reward += reward

    episode_starts = np.array([terminated or truncated])

    if terminated or truncated:
        print(f"Episode reward: {episode_reward:.2f}")
        episode_reward = 0.0
        obs, _ = eval_env.reset()
        lstm_states = None
        episode_starts = np.ones((1,), dtype=bool)

eval_env.close()
